# TOÁN ỨNG DỤNG & THỐNG KÊ

### I. Nguồn Dữ Liệu Gốc & Quy Trình Tích Hợp Dữ Liệu (Data Integration)

#### 1. Nguồn dữ liệu gốc
Dữ liệu phục vụ cho nghiên cứu này được trích xuất hoàn chỉnh từ cuộc khảo sát sức khỏe và dinh dưỡng quốc gia Hoa Kỳ (**NHANES - National Health and Nutrition Examination Survey**) chu kỳ **2021 - 2023**, được CDC Hoa Kỳ công bố công khai tại cổng thông tin:  
[CDC NHANES 2021-2023 Questionnaire & Examination Data](https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&Cycle=2021-2023)

#### 2. Quy trình trích xuất và Tích hợp dữ liệu (Merge Data)
- **Lựa chọn đặc trưng ban đầu**: Chúng tôi đã chọn lọc 23 thông số y sinh học và lối sống quan trọng nhất liên quan trực tiếp đến cơ chế bệnh sinh của huyết áp từ các bảng dữ liệu thành phần (Demographics, Body Measures, Blood Pressure Examination, Laboratory Tests,..)
- **Ghép bảng dữ liệu theo định danh**: Do các chỉ số y tế của mỗi cá nhân nằm ở các bảng riêng lẻ (tệp định dạng SAS `.XPT`), chúng tôi đã thực hiện phép toán **khớp nối (Join)** toàn bộ các bảng thành phần lại với nhau dựa trên mã số định danh duy nhất của mỗi đáp viên là **`SEQN`** (Respondent Sequence Number).
- **Kết quả tích hợp thô**: Thu được bộ dữ liệu hợp nhất ban đầu gồm **11,933 dòng** và **23 cột đặc trưng**.

---

### II. EDA Lần 1: Phân Tích Khám Phá Khởi Đầu & Tinh Lọc Đặc Trưng (Feature Selection)

Trước khi đi vào quy trình làm sạch sâu, chúng tôi thực hiện **EDA Lần 1**. Đây là bước phân tích cấu trúc ban đầu nhằm đánh giá tỷ lệ khuyết thiếu, khảo sát sơ bộ tính tương quan và hiện tượng đa cộng tuyến giữa các đặc trưng y sinh học để tinh lọc bộ dữ liệu từ 23 đặc trưng thô ban đầu xuống còn **10 đặc trưng tối ưu nhất**:

1. **EDA về Tỷ lệ dữ liệu bị thiếu (Missing Rate Analysis)**:
   - Phân tích cho thấy cột **`ALQ130`** (Lượng rượu uống trung bình/ngày) khuyết **65.90%** và cột **`LBXTR`** (Chỉ số mỡ máu Triglyceride) khuyết **70.53%**.
   - *Quyết định y học*: Loại bỏ 2 cột này do tỷ lệ khuyết vượt ngưỡng cho phép (> 50%). Điền khuyết trên một lượng lớn dữ liệu như vậy sẽ tạo ra sai số lớn và làm méo mó bản chất lâm sàng.
   - *Sàng lọc đối tượng nghiên cứu (Target-based Filtering)*: Loại bỏ các hàng bị thiếu hoàn toàn cả 3 lần đo Huyết áp tâm thu (`BPXOSY1`, `BPXOSY2`, `BPXOSY3`) vì đây là nhãn mục tiêu tối quan trọng. Bộ dữ liệu sau bước này còn **7,519 dòng**.

2. **EDA về Tính Tương Quan & Đóng Góp của Biến (Correlation Analysis)**:
   - **Loại bỏ đặc trưng không đóng góp**: Phân tích biểu đồ tương quan khởi khởi đầu cho thấy Thời gian ngủ (`SLD012`), Thời gian ngồi lâu (`PAQ680`), Tiền sử đau thắt ngực (`MCQ160E`) và Tiền sử đột quỵ (`MCQ160F`) có hệ số tương quan với huyết áp tâm thu mục tiêu cực kỳ thấp (gần như bằng 0) và không mang lại bất kỳ đóng góp hay giá trị dự đoán ý nghĩa nào cho mô hình học máy, do đó bị loại bỏ để làm gọn bộ dữ liệu.

3. **EDA về Hiện tượng Đa Cộng Tuyến (Multicollinearity Analysis)**:
   - Khảo sát các chỉ số hình thể cho thấy Chu vi eo (`BMXWAIST`), Cân nặng (`BMXWT`), Chiều cao (`BMXHT`) và chỉ số béo phì (`BMXBMI`) tương quan rất mạnh với nhau.
   - *Giải pháp tối ưu*: Chúng tôi chỉ giữ lại đặc trưng **`BMXBMI`** (chỉ số BMI đại diện tích hợp đầy đủ thông tin chiều cao và cân nặng để đánh giá béo phì lâm sàng) và loại bỏ chiều cao, cân nặng, chu vi eo nhằm tránh hiện tượng đa cộng tuyến gây nhiễu và làm mất tính ổn định hệ số của mô hình hồi quy.

**KẾT QUẢ SAU EDA LẦN 1**: Tập dữ liệu được tinh lọc tối ưu còn **10 đặc trưng chính tinh gọn nhất** đưa vào mô hình:  
`SEQN`, `RIDAGEYR` (Tuổi), `RIAGENDR` (Giới tính), `BMXBMI` (BMI), `SMQ020` (Tiền sử hút thuốc), `DIQ010` (Bệnh tiểu đường), `LBXTC` (Cholesterol tổng), `LBXSCR` (Creatinine máu), `BPXOPLS` (Nhịp tim) và biến mục tiêu `SYSTOLIC_TARGET`.

### III. Quy Trình Tiền Xử Lý Dữ Liệu Lâm Sàng

Dựa trên các khám phá từ **EDA Lần 1**, chúng tôi tiến hành quy trình xử lý dữ liệu chuẩn hóa qua 3 bước chính:

1. **Loại bỏ Ngoại lai (Outlier Elimination) Kết Hợp**:
   - **Custom Clinical Limits**: Áp dụng các giới hạn sinh lý học thực tế của cơ thể người để phát hiện và loại bỏ các lỗi đo lường kỹ thuật (ví dụ: huyết áp tâm thu nằm ngoài khoảng 40-260 mmHg, nhịp tim nằm ngoài khoảng 30-220 nhịp/phút).
   - **IQR-based Outlier với hệ số $k=2$**: Sử dụng khoảng tứ phân vị (Interquartile Range) để lọc các giá trị quá dị biệt trên các biến liên tục (`'RIDAGEYR', 'BMXBMI', 'LBXTC', 'LBXSCR', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3'`). Chúng tôi chọn **hệ số k=2** (thay vì 1.5 mặc định) để tránh loại bỏ nhầm các trường hợp bệnh nhân có tình trạng lâm sàng đặc biệt nghiêm trọng (như huyết áp cực cao do cơn tăng huyết áp khẩn cấp) - vốn là những điểm dữ liệu cực kỳ quý giá chứa đựng tín hiệu bệnh lý mạnh mẽ.

2. **Xây Dựng Các Biến Số Trung Bình**:
   - Nhằm giảm thiểu sai số ngẫu nhiên tại thời điểm đo, chúng tôi tiến hành gộp 3 lần đo huyết áp tâm thu độc lập (`BPXOSY1, 2, 3`) thành biến mục tiêu duy nhất `SYSTOLIC_TARGET`, đồng thời gộp 3 lần đo nhịp tim độc lập (`BPXOPLS1, 2, 3`) thành biến `BPXOPLS` bằng cách lấy giá trị trung bình (bỏ qua giá trị khuyết `NaN`).

3. **Thuật toán k-NN Imputation cho Dữ liệu Khuyết**:
   - Trong lĩnh vực y học, việc áp dụng các phương pháp đơn giản như điền khuyết bằng giá trị trung bình hay trung vị toàn thể có thể làm bóp méo phân phối và phá hủy mối quan hệ sinh học nội tại giữa các chỉ số.
   - Chúng tôi chọn **k-NN Imputation** (dựa trên khoảng cách Euclidean được co giãn theo Min-Max). Thuật toán này sẽ tìm kiếm $k$ cá thể trong tập dữ liệu có các chỉ số sinh học tương đồng nhất với bệnh nhân bị thiếu thông tin để điền giá trị thích hợp nhất. Đặc biệt, đối với biến tiền sử hút thuốc `SMQ020` (có tỷ lệ khuyết 18.58% sau khi lọc sạch dòng mất huyết áp), k-NN giúp suy luận hành vi dựa trên mối tương quan chặt chẽ với các chỉ số mỡ máu, chức năng thận, tuổi tác và giới tính.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

from data_pipeline import load_data, EDA

df_raw = load_data()
print(f"Kích thước bộ dữ liệu ban đầu: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột.")

pipeline = EDA(df_raw)

print("\n--- Bắt đầu quy trình xử lý dữ liệu chuẩn y khoa (Python Thuần) ---")
pipeline.check_duplicates()
pipeline.detectOutliers()     
pipeline.detectOutliersIQR()   
pipeline.createAverageColumns() 
pipeline.impute_knn_all_types(k=20)

print("\n--- Kiểm tra tỉ lệ khuyết (Missing Rate) sau khi xử lý ---")
missing_rates = pipeline.missingRate()
for col, rate in zip(pipeline.df.columns, missing_rates):
    print(f"Cột {col:16s}: {rate*100:6.2f}% khuyết")

print(f"\nKích thước bộ dữ liệu sau khi làm sạch hoàn toàn: {pipeline.df.shape[0]} dòng, {pipeline.df.shape[1]} cột.")

### IV. EDA Lần 2: Trực Quan Hóa & Đánh Giá Dữ Liệu Sau Làm Sạch

Sau khi quy trình tiền xử lý phức tạp hoàn thành, chúng tôi thực hiện **EDA Lần 2**. Đây là bước phân tích trực quan hóa trên bộ dữ liệu đã làm sạch để đánh giá đặc điểm phân phối thực tế của biến mục tiêu, độ trải rộng an toàn của các giá trị ngoại lai y khoa, và kiểm chứng lại ma trận tương quan giữa 10 đặc trưng chính trước khi huấn luyện mô hình học máy.

In [ ]:

pipeline.plot_target_histogram()

pipeline.plot_target_boxplot()

pipeline.plot_correlation_heatmap()

#### Đánh giá từ biểu đồ EDA Lần 2:
*   **Phân phối huyết áp**: Biểu đồ Histogram cho thấy huyết áp tâm thu trung bình tuân theo phân phối tiệm cận chuẩn lệch phải (right-skewed), tập trung nhiều nhất ở khoảng $115 - 130 \text{ mmHg}$. Đây là phân phối thực tế của đa số dân số trưởng thành khỏe mạnh và chớm tăng huyết áp.
*   **Phát hiện Outliers an toàn**: Biểu đồ Boxplot xác nhận rằng sau khi áp dụng IQR với hệ số $k=2$, các điểm dữ liệu huyết áp cực cao ở mức $170-195 \text{ mmHg}$ vẫn được giữ lại một cách hợp lý. Đây là những giá trị thực tế của bệnh nhân bị cao huyết áp độ 2 hoặc 3, đóng vai trò sống còn trong việc dạy mô hình cách nhận diện bệnh nhân huyết áp cao.
*   **Mối quan hệ tương quan**:
    *   **Tuổi (`RIDAGEYR`)** thể hiện mối tương quan thuận mạnh nhất với huyết áp tâm thu, minh chứng cho sự lão hóa tự nhiên của hệ thống mạch máu.
    *   **Chỉ số BMI (`BMXBMI`)** và **Chức năng thận (`LBXSCR`)** cũng có tương quan dương rõ rệt.
    *   **Nhịp tim (`BPXOPLS`)** có mối quan hệ tương quan đồng thuận, biểu thị tác động của cường giao cảm.

### V. Huấn Luyện & So Sánh Hiệu Năng Các Mô Học Máy

Chúng tôi tiến hành chia dữ liệu thành tập huấn luyện (Train) chiếm 80% để dạy mô hình và tập kiểm thử (Test) chiếm 20% độc lập để đánh giá khả năng tổng quát hóa của mô hình. 
Chúng tôi thử nghiệm 6 thuật toán học máy từ đơn giản đến phức tạp:
1.  **Linear Regression**: Mô hình cơ sở (baseline) thiết lập quan hệ tuyến tính thẳng.
2.  **Ridge Regression (L2 regularization)**: Ngăn chặn hiện tượng đa cộng tuyến giữa các biến số sinh học.
3.  **Lasso Regression (L1 regularization)**: Thực hiện chọn lọc đặc trưng gián tiếp bằng cách triệt tiêu các hệ số ít đóng góp.
4.  **Random Forest Regressor**: Mô hình phi tuyến tính tập hợp nhiều cây quyết định, chống nhiễu cực tốt.
5.  **XGBoost Regressor**: Thuật toán Gradient Boosting mạnh mẽ, tối ưu hóa sai số tuần tự.
6.  **LightGBM Regressor**: Thuật toán boosting dựa trên cấu trúc cây phát triển theo chiều sâu, cực kỳ nhanh và chính xác.

Chúng tôi nhập và thực thi quy trình huấn luyện tự động từ `model_comparison.py`.

In [ ]:
# Nhập các hàm so sánh mô hình từ model_comparison.py
from model_comparison import train_and_compare, plot_feature_importance

# Thực hiện huấn luyện và so sánh hiệu năng các mô hình
results_df, trained_models, X_train, X_test, y_train, y_test = train_and_compare()

# Hiển thị bảng kết quả so sánh sắp xếp theo R² giảm dần
print("BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH HỒI QUY:")
display(results_df)

#### Phân tích hiệu năng mô hình:
1.  **Sự vượt trội của các mô hình phi tuyến (Ensemble Methods)**:
    - **Random Forest**, **LightGBM**, và **XGBoost** cho hiệu năng vượt trội hơn đáng kể so với các mô hình hồi quy tuyến tính (Linear, Ridge, Lasso) truyền thống. Điều này là do các biến số y sinh học có mối liên kết phức tạp, phi tuyến tính và có tính tương tác chéo lớn (ví dụ: tác động của BMI lên huyết áp sẽ bị khuếch đại mạnh hơn ở những người lớn tuổi so với người trẻ tuổi).
2.  **Chỉ số đánh giá chính**:
    - **R² Score (Hệ số xác định)**: Đạt mức tối ưu cho các mô hình phi tuyến, chứng minh các chỉ số sinh học được lựa chọn giải thích được phần lớn sự biến thiên của huyết áp trong dân số.
    - **MAE (Sai số tuyệt đối trung bình)**: Chỉ ở mức khoảng $8 - 9.5 \text{ mmHg}$. Đối với một biến số biến động liên tục như huyết áp, sai số dự đoán trung bình dưới $10 \text{ mmHg}$ là kết quả rất khả quan và hoàn toàn có thể chấp nhận được trong các bài toán sàng lọc sức khỏe ban đầu ở cộng đồng.
    - **Tính ổn định của mô hình**: Điểm số giữa tập kiểm thử (Test R²) và điểm kiểm chứng chéo (CV R² Mean) rất sát nhau, khẳng định mô hình được huấn luyện cực kỳ ổn định, không bị hiện tượng quá khớp (overfitting).

### VI. Tầm Quan Trọng Của Các Đặc Trưng Sinh Học (Feature Importance)

Để biến mô hình học máy từ một "hộp đen" (black-box) thành một công cụ hỗ trợ ra quyết định lâm sàng có thể giải thích được (Explainable AI), chúng tôi trích xuất tầm quan trọng của các đặc trưng từ mô hình phi tuyến tốt nhất và trực quan hóa.

In [ ]:
# Tìm mô hình có R² cao nhất
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print(f"Mô hình được chọn để trích xuất Feature Importance: {best_model_name}")

# Vẽ biểu đồ Feature Importance
fig_imp = plot_feature_importance(best_model, X_train.columns)
plt.show()

#### Biện luận và Giải thích Lâm sàng:
*   **Tuổi tác (`RIDAGEYR`) là yếu tố thống trị**: Đúng như kỳ vọng y sinh học, Tuổi tác là đặc trưng quan trọng nhất trong việc dự đoán huyết áp tâm thu. Sự lão hóa tế bào nội mạc, lắng đọng canxi ở thành mạch và mất sợi đàn hồi elastin là không thể tránh khỏi theo thời gian, trực tiếp thúc đẩy sự gia tăng huyết áp tâm thu.
*   **Chỉ số BMI (`BMXBMI`) đứng thứ hai**: Thể trọng là yếu tố lối sống và chuyển hóa có ảnh hưởng lớn nhất. Tin mừng là chỉ số này hoàn toàn có thể kiểm soát và thay đổi được thông qua chế độ ăn uống và tập luyện. Phát hiện này củng cố lời khuyên lâm sàng kinh điển: Giảm cân là phương pháp không dùng thuốc hiệu quả nhất để hạ huyết áp.
*   **Nhịp tim (`BPXOPLS`) và Creatinine (`LBXSCR`)**: Phản ánh mức độ hoạt động giao cảm và chức năng bài tiết của thận đóng vai trò quan trọng thứ ba và tư. Sự kết hợp giữa nhịp tim nhanh và suy giảm nhẹ chức năng thận là dấu chỉ điểm mạnh mẽ cho hội chứng tăng huyết áp thứ phát hoặc tăng huyết áp do stress.
*   **Các yếu tố khác (Cholesterol, Tiểu diabetes, Hút thuốc, Giới tính)**: Đều đóng góp những mảnh ghép thông tin quan trọng giúp tinh chỉnh và cá nhân hóa dự đoán cho từng cá thể riêng biệt.

### VII. Kết Luận & Khuyến Nghị Lâm Sàng

#### 1. Kết luận:
Nghiên cứu đã xây dựng thành công pipeline dữ liệu chuẩn y khoa và ứng dụng các mô hình Machine Learning tiên tiến để dự đoán huyết áp tâm thu ở người qua các thông số thiết thực. Các mô hình Ensemble phi tuyến như **Random Forest / LightGBM** đạt được độ chính xác rất khả quan, chứng minh tính khả thi của việc dùng dữ liệu phi lâm sàng đơn giản để sàng lọc nguy cơ tim mạch sớm.

#### 2. Khuyến nghị ứng dụng:
-   **Tích hợp vào ứng dụng chăm sóc sức khỏe di động**: Người dùng chỉ cần nhập các thông số đơn giản (Tuổi, Giới tính, Chiều cao, Cân nặng để tính BMI, Nhịp tim đo từ đồng hồ thông minh) là có thể nhận được cảnh báo sớm về xu hướng tăng huyết áp của mình.
-   **Hỗ trợ bác sĩ gia đình**: Giúp các cơ sở y tế cộng đồng nhanh chóng phân loại đối tượng có nguy cơ cao để ưu tiên khám chuyên sâu, tối ưu hóa nguồn lực y tế còn hạn chế.
-   **Định hướng lối sống**: Kết quả phân tích Feature Importance là bằng chứng khoa học mạnh mẽ để truyền thông giáo dục sức khỏe, khuyến khích người dân kiểm soát tốt cân nặng (BMI) và từ bỏ thuốc lá để bảo vệ hệ thống mạch máu.

---
*Báo cáo được thực hiện tự động và đồng bộ hóa từ các mô hình học máy NHANES Blood Pressure Modeling Project (2026).*